In [1]:
import re
from typing import List

import pandas as pd
import requests
from opencc import OpenCC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "qwen2.5:7b-instruct"
CSV_PATH = r"C:\Users\Yun-chiao\Downloads\期中報告\complaints_final.csv"   # 改成你的路徑

cc = OpenCC("s2t")

In [4]:
df = pd.read_csv(CSV_PATH)

required_cols = [
    "Complaint ID",
    "Issue",
    "Sub-issue",
    "Consumer complaint narrative in Chinese",
    "Solution in Chinese"
]

df = df[required_cols].copy()
df = df.fillna("")

print("資料筆數：", len(df))
df.head()

資料筆數： 86


,Complaint ID,Issue,Sub-issue,Consumer complaint narrative in Chinese,Solution in Chinese
0,17987894,Managing an account,Deposits and withdrawals,申訴人表示原應匯入 WELLS FARGO & COMPANY 帳戶的 ACH／直接存款，金...,建議處理方案（專業版）：1. 先以 ACH / direct deposit trace 流...
1,17939613,Managing an account,Deposits and withdrawals,"申訴人表示於 2025-12-01 在 CITIBANK, N.A. 的 ATM 辦理存款時...",建議處理方案（專業版）：1. 立即以案件編號啟動 ATM 爭議調查，調閱 ATM 電子日誌、...
2,17877488,Managing an account,Deposits and withdrawals,申訴人表示其在 WELLS FARGO & COMPANY 的 checking accou...,建議處理方案（專業版）：1. 先由對應營運/客服後台重建案件時間軸，整理客戶主張、交易紀錄、...
3,17998026,Managing an account,Deposits and withdrawals,"申訴人表示 BANK OF AMERICA, NATIONAL ASSOCIATION 以異...",建議處理方案（專業版）：1. 由風控/合規團隊確認限制原因（身分驗證、可疑交易、退票風險、K...
4,18049411,Managing an account,Deposits and withdrawals,"申訴人表示其在 BANK OF AMERICA, NATIONAL ASSOCIATION ...",建議處理方案（專業版）：1. 先由對應營運/客服後台重建案件時間軸，整理客戶主張、交易紀錄、...


In [5]:
stopwords = {
    "的", "了", "呢", "嗎", "我", "想", "請問", "一下", "如何", "怎麼", "要", "該", "是不是",
    "a", "an", "the", "is", "are", "to", "for", "and", "or", "in", "on", "with"
}

def normalize_text(text: str) -> str:
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\u4e00-\u9fff\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text

def tokenize(text: str) -> List[str]:
    text = normalize_text(text)

    zh_blocks = re.findall(r"[\u4e00-\u9fff]+", text)
    en_tokens = re.findall(r"[a-zA-Z0-9_]+", text)

    zh_tokens = []
    for block in zh_blocks:
        block = block.strip()
        if len(block) == 1:
            zh_tokens.append(block)
        else:
            for n in [2, 3, 4]:
                for i in range(len(block) - n + 1):
                    zh_tokens.append(block[i:i+n])

    all_tokens = zh_tokens + en_tokens
    return [t for t in all_tokens if t not in stopwords and len(t) > 1]

def normalize_solution_text(text: str) -> str:
    text = str(text).lower().strip()
    text = re.sub(r"\$[\d,]+(\.\d+)?", "金額", text)
    text = re.sub(r"\d+(\.\d+)?", "數字", text)
    text = re.sub(r"[^\w\u4e00-\u9fff\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def solution_similarity(sol1: str, sol2: str) -> float:
    tokens1 = set(tokenize(normalize_solution_text(sol1)))
    tokens2 = set(tokenize(normalize_solution_text(sol2)))

    if not tokens1 or not tokens2:
        return 0.0

    return len(tokens1 & tokens2) / max(len(tokens1), 1)

def keyword_overlap_score(question: str, case_doc: str) -> float:
    q_tokens = set(tokenize(question))
    c_tokens = set(tokenize(case_doc))

    if not q_tokens or not c_tokens:
        return 0.0

    return len(q_tokens & c_tokens) / max(len(q_tokens), 1)

In [6]:
def build_case_document(row: pd.Series) -> str:
    parts = [
        str(row.get("Issue", "")),
        str(row.get("Sub-issue", "")),
        str(row.get("Consumer complaint narrative in Chinese", "")),
        str(row.get("Consumer complaint narrative in Chinese", "")),
        str(row.get("Solution in Chinese", "")),
        str(row.get("Solution in Chinese", ""))
    ]
    return "\n".join([p for p in parts if p.strip() != ""])

df["case_document"] = df.apply(build_case_document, axis=1)
df[["Complaint ID", "case_document"]].head()

,Complaint ID,case_document
0,17987894,Managing an account\nDeposits and withdrawals\...
1,17939613,Managing an account\nDeposits and withdrawals\...
2,17877488,Managing an account\nDeposits and withdrawals\...
3,17998026,Managing an account\nDeposits and withdrawals\...
4,18049411,Managing an account\nDeposits and withdrawals\...


In [ ]:
def find_best_category(question: str, df: pd.DataFrame): #demo不使用
    category_df = (
        df[["Issue", "Sub-issue"]]
        .fillna("")
        .astype(str)
        .drop_duplicates()
        .reset_index(drop=True)
    )

    category_df["category_text"] = category_df["Issue"] + " | " + category_df["Sub-issue"]
    corpus = category_df["category_text"].tolist() + [question]

    vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(2, 4),
        min_df=1
    )
    matrix = vectorizer.fit_transform(corpus)

    category_vectors = matrix[:-1]
    question_vector = matrix[-1]

    similarities = cosine_similarity(question_vector, category_vectors).flatten()
    best_idx = similarities.argmax()

    best_issue = category_df.loc[best_idx, "Issue"]
    best_sub_issue = category_df.loc[best_idx, "Sub-issue"]

    return best_issue, best_sub_issue

In [ ]:
#核心檢索函式
#先自定義 > 相似度 0.7 ; 關鍵詞 0.3
def rank_cases(question: str, df: pd.DataFrame, top_k: int = 3) -> pd.DataFrame:
    
    #如果沒有找到case_document，主動中止程式，並回報一個錯誤訊息
    if "case_document" not in df.columns:
        raise ValueError("df 裡沒有 case_document 欄位，請先建立它。")

    # 不使用 Issue / Sub-issue 分類(def find_best_category)，直接在全部案例中檢索
    filtered_df = df.copy()

    docs = filtered_df["case_document"].tolist() #把case文字變成一個list
    corpus = docs + [question]

    #TF-IDF 向量: 用來從一段文字/一個語料庫中，給越重要的字詞/文檔，越高的加權分數。
    #字詞的重要性隨著 在文本出現的頻率越高則越高；在不同文本檔案間出現的次數越高則反而降低
    vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        #char_wb 以字元為單位切片，但會考慮單字邊界（word boundary），這對中英混合、中文、拼寫差異都比較有彈性
        ngram_range=(2, 4),
        #要取 2 到 4 個字元的片段，可以抓到局部字串相似性
        min_df=1
        #表示只要某個字元片段出現過 1 次，就保留
    )
    matrix = vectorizer.fit_transform(corpus)
    #用剛才TfidfVectorizer的定義遍歷資料，一邊把文字轉成 tf-idf 矩陣，通常用在第一次處理訓練資料 / 全部語料 的時候

    case_vectors = matrix[:-1] #把case、question分開
    question_vector = matrix[-1]

    #計算 問題向量 和 每一筆案例向量 的 cosine similarity（餘弦相似度）也就是兩個向量是不是在重要特徵上指向差不多的方向
    #cosine分數通常介於 0 ~ 1 之間，越大表示越相似
    # .flatten()把結果攤平成一維陣列，方便後面直接用索引取值。
    similarities = cosine_similarity(question_vector, case_vectors).flatten()

    rows = []

    #df.iterrows()會取出index + row，但是(_, row)代表只保留row
    #df的index做過篩選/刪除/排序/合併，pandas 通常會保留原本的 index，不會自動重編
    # enumerate 幫每列加上一個從 0 開始的編號 i，這個 i 可以對應到前面 similarities[i]
    for i, (_, row) in enumerate(filtered_df.iterrows()):
        sim = float(similarities[i])
        keyword_score = keyword_overlap_score(question, row["case_document"])
        final_score = (sim * 0.7) + (keyword_score * 0.3)

        rows.append({
            "Complaint ID": row.get("Complaint ID", ""),
            "Issue": row.get("Issue", ""),
            "Sub-issue": row.get("Sub-issue", ""),
            "Consumer complaint narrative in Chinese": row.get("Consumer complaint narrative in Chinese", ""),
            "Solution in Chinese": row.get("Solution in Chinese", ""),
            "similarity_score": sim,
            "keyword_score": keyword_score,
            "final_score": final_score
        })

    result_df = pd.DataFrame(rows).sort_values("final_score", ascending=False).reset_index(drop=True)

    # 把相似解法去重
    unique_solutions = []
    for _, row in result_df.iterrows():
        current_solution = row["Solution in Chinese"]

        is_similar = False
        for saved_row in unique_solutions: #只有第一圈是空的，之後會逐漸壯大
            sim_score = solution_similarity(current_solution, saved_row["Solution in Chinese"])
            if sim_score > 0.75:
                is_similar = True
                break

        if not is_similar:
            unique_solutions.append(row)

        if len(unique_solutions) >= top_k:
            break

    return pd.DataFrame(unique_solutions).reset_index(drop=True)

In [26]:
def build_prompt(question: str, top_results: pd.DataFrame) -> str:
    solution_blocks = []

    for i, row in top_results.iterrows():
        solution = str(row["Solution in Chinese"]).replace("建議處理方案（專業版）：", "").strip()
        solution_blocks.append(f"方案{i+1}：{solution}")

    solutions_text = "\n\n".join(solution_blocks)

    prompt = f"""
請只使用繁體中文回答，禁止使用任何簡體字。

你是台灣公司內部的資深 PM 助手，要幫一位沒有經驗的新手 PM 判斷問題。

使用者問題：
{question}

系統找到的參考方案：
{solutions_text}

請你根據上面的方案，整理成自然、白話、像真人主管帶新人的回答。

規則：
1. 先用 2 句話理解問題。
2. 再整理成 3 個排查方向。
3. 每個方向要講清楚先做什麼。
4. 只採用與問題主題一致的方案。
5. 如果某個方案明顯屬於別的題型，請直接忽略，不要寫進回答。
6. 若高相關方向不足三個，寧可只回答兩個，也不要硬湊不相關內容。
7. 對於「帳戶被限制、無法提領資金」這類問題，優先從：
   - 限制原因
   - 補件與審查進度
   - 資金返還或解除限制流程
   這三個方向整理。
8. 不要逐字抄原文，但要保留具體動作與關鍵詞，例如付款 ID、trace number、入帳紀錄、清算結果、沖回、補件、風控、合規、解除限制、返還餘額。
9. 請使用台灣常用繁體中文詞彙，例如「帳戶、後台、聯絡、資料、流程」，不要使用「賬戶、後臺、聯繫」這類中國用語。

最後請明確告訴我：第一步先查什麼，原因是什麼。
"""
    return prompt.strip()

In [ ]:
def ask_ollama(prompt: str, model: str = OLLAMA_MODEL, timeout: int = 300) -> str:
    payload = {
        "model": model,
        "prompt": prompt,
        "system": "你是台灣的資深產品經理助理。你必須只使用繁體中文回答，禁止使用任何簡體字。",
        "stream": False
    }

    try:
        response = requests.post(OLLAMA_URL, json=payload, timeout=timeout)
        response.raise_for_status()

        data = response.json()
        result = data.get("response", "").strip()
        result = cc.convert(result)
        return result

    except requests.exceptions.ConnectionError:
        return "無法連線到 Ollama。請先確認 Ollama 應用程式或 serve 服務有啟動。"
    except requests.exceptions.Timeout:
        return "Ollama 回應逾時。"
    except requests.exceptions.HTTPError:
        return f"Ollama HTTP 錯誤：{response.text}"
    except Exception as e:
        return f"Ollama 呼叫失敗：{e}"

In [28]:
def pm_ai_assistant(question: str, df: pd.DataFrame, top_k: int = 3):
    top_results = rank_cases(question, df, top_k=top_k)
    prompt = build_prompt(question, top_results)
    answer = ask_ollama(prompt)
    return answer, top_results

In [29]:
question = "消費者反映交易失敗，但帳戶還是被扣款，PM 應該先查什麼？"

In [ ]:
top_results = rank_cases(question, df, top_k=3)
top_results

In [ ]:
for i, row in top_results.iterrows():
    solution = str(row["Solution in Chinese"]).replace("建議處理方案（專業版）：", "").strip()
    print("=" * 80)
    print(f"方案 {i+1}")
    print(solution)
    print()

In [33]:
answer, top_results = pm_ai_assistant(question, df, top_k=3)
print(answer)

好的，我們來整理一下：

1. 首先要了解的是交易失敗的具體原因，所以第一步是確認付款授權與清算結果。
2. 然後需要檢查是否系統出現誤判或是扣款問題，這時可以比對入帳紀錄及交易序號。
3. 最後若是涉及第三方支付或外部商戶的情況，則需確認相關銀行或第三方平臺的處理流程。

根據以上三個方向，第一步先查付款授權、取消回應碼以及清算結果。原因是在交易失敗的情況下，這些資料能幫助我們初步判斷是否真的存在扣款問題或是系統誤判，進而避免不必要的衝突與損失。
